# 03 · Cell-type annotation
**Input :** `data/processed/adata_clustered.h5ad`  
**Output:** `data/processed/adata_annotated.h5ad`

Uses CellTypist, ScType (via R), and SingleR (via R) in a 4-way majority vote.
ScType gets double weight for liver parenchymal types (Hepatocyte, Fibroblast, Endothelial).

**Requires:** R 4.3+, Seurat, SingleR, celldex — run `Rscript env/r_packages.R` first.

**Run order:** 02 → **03** → 04 → 05


In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    for p in [start, *start.parents]:
        if (p / "paths.py").exists():
            return p
    raise FileNotFoundError("paths.py not found — run: python scripts/data_download.py")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "scripts"))

from paths import REPO_ROOT, RAW_DIR, PROC_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR
print(f"Repo root : {REPO_ROOT}")
print(f"Raw data  : {RAW_DIR}")

In [ ]:
import scanpy as sc
from scrna_functions import (run_celltypist, prep_seurat_object, pull_r_col,
                              marker_score_clusters, majority_vote, save_adata,
                              MARKER_SETS)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
LEIDEN_COL = "leiden_res_0.50"   # clustering resolution to annotate

## 1 · Load

In [ ]:
adata = sc.read(str(PROC_DIR / "adata_clustered.h5ad"))
print(adata)

## 2 · R environment (run once per session)

In [ ]:
import os, sys

# ── R environment — update R_HOME if needed ───────────────────────────────────
# macOS  : /Library/Frameworks/R.framework/Resources
# Linux  : /usr/lib/R
# Windows: r"C:\Program Files\R\R-4.5.3"
R_HOME = os.environ.get("R_HOME", "")
if R_HOME:
    os.environ["R_HOME"] = R_HOME
    if sys.platform == "win32":
        os.environ["PATH"] = os.path.join(R_HOME,"bin","x64") + ";" + os.environ["PATH"]

%load_ext rpy2.ipython
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import default_converter
import anndata2ri
print("R environment ready")

## 3 · CellTypist

In [ ]:
adata = run_celltypist(adata)
sc.pl.umap(adata, color=["celltypist_coarse","celltypist_fine"],
           frameon=False, wspace=1)

## 4 · ScType (liver-specific database)

In [ ]:
%%R
library(Seurat); library(dplyr); library(HGNChelper); library(openxlsx)
source("https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/R/gene_sets_prepare.R")
source("https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/R/sctype_score_.R")
gs_list <- gene_sets_prepare(
    "https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/ScTypeDB_full.xlsx",
    "Liver")

In [ ]:
adata_seurat = prep_seurat_object(adata, ro, default_converter, anndata2ri)

In [ ]:
%%R
expr_matrix <- assay(adata_seurat, "logcounts")
es_max <- sctype_score(scRNAseqData=expr_matrix, scaled=TRUE,
                       gs=gs_list$gs_positive, gs2=gs_list$gs_negative)
clusters <- colData(adata_seurat)$leiden_res_0.50
cl_results <- lapply(unique(clusters), function(cl) {
    cells <- which(clusters == cl)
    es    <- sort(rowSums(es_max[, cells, drop=FALSE]), decreasing=TRUE)
    data.frame(cluster=cl, cell_type=names(es)[1], score=es[1])
}) |> dplyr::bind_rows()
cluster_to_type <- setNames(cl_results$cell_type, cl_results$cluster)
colData(adata_seurat)$sctype_cell_type <- cluster_to_type[as.character(clusters)]
print(cl_results)

In [ ]:
adata = pull_r_col(adata, ro, default_converter, pandas2ri,
                   col_name="sctype_cell_type", obs_col="sctype_cell_type")
sc.pl.umap(adata, color="sctype_cell_type", legend_loc="on data",
           legend_fontsize=8, frameon=False)

## 5 · SingleR (HPCA reference)

In [ ]:
%%R
library(SingleR); library(celldex)
ref_hpca  <- HumanPrimaryCellAtlasData()
pred_hpca <- SingleR(test=adata_seurat, ref=ref_hpca,
                     labels=ref_hpca$label.main)
colData(adata_seurat)$SingleR_HPCA <- pred_hpca$pruned.labels

In [ ]:
adata = pull_r_col(adata, ro, default_converter, pandas2ri,
                   col_name="SingleR_HPCA", obs_col="SingleR_HPCA")

## 6 · Marker scoring & majority vote

In [ ]:
score_df = marker_score_clusters(adata, leiden_col=LEIDEN_COL,
                                  marker_sets=MARKER_SETS)

In [ ]:
adata, vote_df = majority_vote(adata, score_df, leiden_col=LEIDEN_COL)
sc.pl.umap(adata,
           color=["manual_celltype","leiden_res_0.50","sample"],
           legend_loc="on data", legend_fontsize=8,
           frameon=False, ncols=3)

## 7 · Save

In [ ]:
save_adata(adata, PROC_DIR / 'adata_annotated.h5ad')